In [1]:
from scipy import stats
import statsmodels.api as sm
from torch import distributions, manual_seed, set_grad_enabled, tensor

import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

In [2]:
COLORS = [
    '#00B0F0',
    '#FF0000'
]

In [3]:
set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

# Chapter 03

In this chapter we focus on building an intuition between causal concepts and linear regression. Next, we move to a comparison between observational and interventional distributions and show why and how they might be different. Finally, we introduce the concept of structural models. Structural models will naturally lead us to the concept of graphical models in the next chapter.

## Regression

Let's build a model according to the followingh specification:

$$\Large \hat{y}_i = 1.12 + 0.93x_i + 0.5 \epsilon_i$$ 

where $\epsilon \sim \mathcal{N}(0, 1)$

In [4]:
# Set the seed for reproducibility
manual_seed(45)

# No. of samples
N_SAMPLES = 5000

# Define true model parameters
alpha = 1.12
beta = 0.93
epsilon = distributions.Normal(tensor([0.0]), tensor([1.0])).sample((N_SAMPLES,))

# Generate X
X = distributions.Normal(tensor([0.0]), tensor([1.0])).sample((N_SAMPLES,))

# Compute Y
y = alpha + beta * X + 0.5 * epsilon

# Statsmodel models require us to add constant (used to evaluate intercept)
X = sm.add_constant(X)

print(X[:5, :])

[[ 1.         -0.49665338]
 [ 1.          1.43214405]
 [ 1.          1.03852963]
 [ 1.          0.62991685]
 [ 1.         -0.36702567]]


The `sm.add_constant` accounts for the fact that the relation between $X$ and $Y$ is not linear, but affine.

$$Y = \begin{bmatrix}\alpha & 0 \\ 0 & \beta\end{bmatrix} \begin{bmatrix}1 \\ X\end{bmatrix}$$

In [5]:
# Instantiate the model and fit it
model = sm.OLS(y.numpy(), X)
fitted_model = model.fit()

# Print results summary
print(fitted_model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.775
Model:                            OLS   Adj. R-squared:                  0.775
Method:                 Least Squares   F-statistic:                 1.724e+04
Date:                Sat, 07 Mar 2026   Prob (F-statistic):               0.00
Time:                        16:35:22   Log-Likelihood:                -3682.6
No. Observations:                5000   AIC:                             7369.
Df Residuals:                    4998   BIC:                             7382.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.1165      0.007    156.177      0.0